# Laboratorio 4 — Cuaderno 3: análisis temporal

**Ejercicio 4.** Cómo cambia la cianobacteria a lo largo del tiempo en cada lago.

Tenemos 11 fechas por lago repartidas entre enero de 2025 y julio de 2026. No son fechas
equiespaciadas —dependen de cuándo el satélite pasó sin nubes— así que hay que tener cuidado
al hablar de tendencias: los huecos no son iguales entre un punto y el siguiente.

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from src import config, datos, graficos
from src import indices as ix

resumen = datos.tabla_resumen()
resumen = resumen.sort_values(["lago", "fecha"]).reset_index(drop=True)
print(f"{len(resumen)} escenas cargadas")

COLOR = {"Atitlan": "#1f6f8b", "Amatitlan": "#c1272d"}

## 4.1 Índice promedio de cianobacteria por lago y por fecha

Para cada escena promediamos la clorofila-a sobre todos los píxeles de agua que no estaban
tapados por nube. Reportamos también la mediana y el percentil 90, porque el promedio de una
distribución muy sesgada puede esconder lo que pasa en las zonas peores del lago.

In [ ]:
tabla = resumen[[
    "lago", "fecha", "chl_media", "chl_mediana", "chl_p90", "chl_max",
    "pct_alto", "area_agua_km2", "cobertura_valida_pct", "nubosidad_pct",
]].copy()
tabla["fecha"] = tabla["fecha"].dt.strftime("%Y-%m-%d")

for lago in ["Atitlan", "Amatitlan"]:
    print(f"\n{config.NOMBRE_LARGO[lago]}")
    print(tabla[tabla["lago"] == lago].drop(columns="lago").round(2).to_string(index=False))

In [ ]:
estadisticas = resumen.groupby("lago").agg(
    n_escenas=("chl_media", "size"),
    chl_media=("chl_media", "mean"),
    chl_min=("chl_media", "min"),
    chl_max=("chl_media", "max"),
    chl_std=("chl_media", "std"),
    pct_alto_medio=("pct_alto", "mean"),
    pct_alto_max=("pct_alto", "max"),
).round(2)
estadisticas

## 4.2 Evolución temporal

Cada lago va en su propio panel porque los rangos de concentración son muy distintos y
compartir eje aplastaría al de valores más bajos.

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = resumen[resumen["lago"] == lago]

    eje.plot(sub["fecha"], sub["chl_media"], "o-", color=COLOR[lago],
             linewidth=2, markersize=7, label="Media del lago", zorder=3)
    eje.plot(sub["fecha"], sub["chl_mediana"], "s--", color=COLOR[lago],
             alpha=0.5, markersize=5, label="Mediana", zorder=2)
    eje.fill_between(sub["fecha"], sub["chl_mediana"], sub["chl_p90"],
                     color=COLOR[lago], alpha=0.15,
                     label="Mediana a percentil 90", zorder=1)

    promedio = sub["chl_media"].mean()
    eje.axhline(promedio, color="gray", linestyle=":", linewidth=1.2,
                label=f"Promedio del período ({promedio:.1f} µg/L)")

    eje.set_title(f"{config.NOMBRE_LARGO[lago]}", fontsize=13, loc="left")
    eje.set_ylabel("Clorofila-a (µg/L)")
    eje.legend(fontsize=9, loc="upper left")
    eje.grid(alpha=0.3)

ejes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ejes[-1].set_xlabel("Fecha")
fig.suptitle("Evolución temporal del índice de cianobacteria", fontsize=15)
fig.tight_layout()
plt.show()

### Los dos lagos en la misma escala

Para dimensionar la diferencia entre ellos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

for lago in ["Atitlan", "Amatitlan"]:
    sub = resumen[resumen["lago"] == lago]
    ejes[0].plot(sub["fecha"], sub["chl_media"], "o-", color=COLOR[lago],
                 linewidth=2, markersize=6, label=config.NOMBRE_LARGO[lago])
    ejes[1].plot(sub["fecha"], sub["pct_alto"], "o-", color=COLOR[lago],
                 linewidth=2, markersize=6, label=config.NOMBRE_LARGO[lago])

ejes[0].set_title("Clorofila-a media (escala común)", loc="left")
ejes[0].set_ylabel("Clorofila-a (µg/L)")
ejes[1].set_title(f"Superficie con floración intensa (> {ix.UMBRAL_ALTO_CHL:.0f} µg/L)",
                  loc="left")
ejes[1].set_ylabel("% del espejo de agua")

for eje in ejes:
    eje.legend(fontsize=9)
    eje.grid(alpha=0.3)
    eje.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    eje.tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()

## 4.3 Picos de floración y fechas críticas

Marcamos como **fecha crítica** una escena que cumple al menos uno de estos dos criterios:

1. Su media está más de una desviación estándar por encima del promedio del lago en el período
   (es decir, se sale claramente del comportamiento típico de ese lago).
2. Más del 10 % del espejo de agua supera el umbral de alerta de 50 µg/L.

Adicionalmente identificamos los **máximos locales**: fechas cuyo valor es mayor que el de la
escena anterior y el de la siguiente.

In [ ]:
def fechas_criticas(sub):
    valores = sub["chl_media"].to_numpy()
    media, desv = np.nanmean(valores), np.nanstd(valores)
    umbral = media + desv

    sub = sub.copy()
    sub["z"] = (valores - media) / desv if desv > 0 else 0.0
    sub["sobre_umbral"] = valores > umbral
    sub["extension_alta"] = sub["pct_alto"] > 10

    # Máximo local: mayor que sus dos vecinos temporales.
    es_pico = np.zeros(len(valores), dtype=bool)
    for i in range(1, len(valores) - 1):
        if valores[i] > valores[i - 1] and valores[i] > valores[i + 1]:
            es_pico[i] = True
    if len(valores) > 1:
        es_pico[0] = valores[0] > valores[1]
        es_pico[-1] = valores[-1] > valores[-2]
    sub["pico_local"] = es_pico
    sub["critica"] = sub["sobre_umbral"] | sub["extension_alta"]
    return sub, umbral

marcadas = {}
for lago in ["Atitlan", "Amatitlan"]:
    sub, umbral = fechas_criticas(resumen[resumen["lago"] == lago])
    marcadas[lago] = sub
    print(f"\n{'='*62}\n{config.NOMBRE_LARGO[lago]}")
    print(f"Promedio del período: {sub['chl_media'].mean():.1f} µg/L | "
          f"umbral de anomalía: {umbral:.1f} µg/L")

    criticas = sub[sub["critica"]]
    if len(criticas):
        print(f"\nFechas críticas ({len(criticas)}):")
        for _, f in criticas.iterrows():
            motivos = []
            if f["sobre_umbral"]:
                motivos.append(f"anomalía alta (z={f['z']:+.2f})")
            if f["extension_alta"]:
                motivos.append(f"{f['pct_alto']:.1f}% del lago sobre 50 µg/L")
            print(f"  {f['fecha']:%Y-%m-%d}  media {f['chl_media']:6.1f} µg/L  "
                  f"máx {f['chl_max']:7.1f}  →  {'; '.join(motivos)}")
    else:
        print("\nNinguna fecha supera los criterios de anomalía.")

    picos = sub[sub["pico_local"]]
    print(f"\nMáximos locales: "
          + ", ".join(f"{f['fecha']:%Y-%m-%d} ({f['chl_media']:.1f})"
                      for _, f in picos.iterrows()))

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = marcadas[lago]
    eje.plot(sub["fecha"], sub["chl_media"], "o-", color=COLOR[lago],
             linewidth=2, markersize=6, zorder=2)

    criticas = sub[sub["critica"]]
    eje.scatter(criticas["fecha"], criticas["chl_media"], s=220, facecolors="none",
                edgecolors="#d95f02", linewidths=2.5, zorder=4,
                label="Fecha crítica")
    for _, f in criticas.iterrows():
        eje.annotate(f"{f['fecha']:%d %b %y}\n{f['chl_media']:.1f} µg/L",
                     (f["fecha"], f["chl_media"]),
                     textcoords="offset points", xytext=(0, 16),
                     ha="center", fontsize=8, color="#d95f02", fontweight="bold")

    media = sub["chl_media"].mean()
    desv = sub["chl_media"].std()
    eje.axhline(media, color="gray", linestyle=":", linewidth=1)
    eje.axhline(media + desv, color="#d95f02", linestyle="--", linewidth=1,
                label="Umbral de anomalía (media + 1 desv.)")

    eje.set_title(config.NOMBRE_LARGO[lago], fontsize=13, loc="left")
    eje.set_ylabel("Clorofila-a (µg/L)")
    eje.legend(fontsize=9, loc="upper left")
    eje.grid(alpha=0.3)
    eje.margins(y=0.22)

ejes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.suptitle("Picos de floración y fechas críticas", fontsize=15)
fig.tight_layout()
plt.show()

## 4.4 Patrones temporales

### Estacionalidad

Guatemala tiene dos estaciones marcadas: **seca**, de noviembre a abril, y **lluviosa**, de mayo
a octubre. Cada una empuja la floración en direcciones distintas:

- En la **estación seca** el agua se estanca, hay más horas de sol y menos mezcla vertical, lo
  que favorece que la cianobacteria se acumule en la superficie.
- En la **estación lluviosa** entra más nutriente arrastrado desde las cuencas —fertilizante,
  aguas residuales, sedimento— pero también hay más nubes, más mezcla y más dilución.

Agrupamos las fechas disponibles por estación y por mes para ver hacia dónde se inclina cada lago.

In [ ]:
def estacion(fecha):
    return "Seca (nov-abr)" if fecha.month in (11, 12, 1, 2, 3, 4) else "Lluviosa (may-oct)"

resumen["estacion"] = resumen["fecha"].apply(estacion)
resumen["mes"] = resumen["fecha"].dt.month
resumen["anio"] = resumen["fecha"].dt.year

por_estacion = resumen.groupby(["lago", "estacion"]).agg(
    n=("chl_media", "size"),
    chl_media=("chl_media", "mean"),
    chl_max=("chl_media", "max"),
    pct_alto=("pct_alto", "mean"),
).round(2)
por_estacion

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = resumen[resumen["lago"] == lago]
    grupos = [sub[sub["estacion"] == e]["chl_media"].dropna().values
              for e in ["Seca (nov-abr)", "Lluviosa (may-oct)"]]
    partes = eje.boxplot(grupos, tick_labels=["Seca\n(nov-abr)", "Lluviosa\n(may-oct)"],
                         patch_artist=True, widths=0.55)
    for caja, c in zip(partes["boxes"], ["#e8a33d", "#4a7c9e"]):
        caja.set_facecolor(c); caja.set_alpha(0.6)
    for i, g in enumerate(grupos, start=1):
        eje.scatter(np.random.normal(i, 0.05, len(g)), g, color="black",
                    s=28, zorder=3, alpha=0.75)
    eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=12)
    eje.set_ylabel("Clorofila-a media (µg/L)")
    eje.grid(alpha=0.3, axis="y")

fig.suptitle("Distribución del índice por estación del año", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
fig, eje = plt.subplots(figsize=(12, 5))

for lago in ["Atitlan", "Amatitlan"]:
    sub = resumen[resumen["lago"] == lago].sort_values("mes")
    eje.plot(sub["mes"], sub["chl_media"], "o", color=COLOR[lago], markersize=9,
             label=config.NOMBRE_LARGO[lago], alpha=0.85)
    for _, f in sub.iterrows():
        eje.annotate(f"{f['fecha']:%y}", (f["mes"], f["chl_media"]),
                     textcoords="offset points", xytext=(7, -3), fontsize=7,
                     color=COLOR[lago])

eje.axvspan(4.5, 10.5, color="#4a7c9e", alpha=0.10)
eje.text(7.5, eje.get_ylim()[1] * 0.95, "estación lluviosa",
         ha="center", fontsize=9, color="#33637f")
eje.set_xticks(range(1, 13))
eje.set_xticklabels(["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                     "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"])
eje.set_xlabel("Mes")
eje.set_ylabel("Clorofila-a media (µg/L)")
eje.set_title("Índice de cianobacteria por mes del año (etiqueta = año)", loc="left")
eje.legend()
eje.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Variabilidad y relación con la nubosidad

Antes de atribuir cualquier subida a un fenómeno ecológico conviene descartar que sea un
artefacto de la imagen. Si las fechas con más nubes fueran justo las de índice más alto,
tendríamos un problema de medición y no una floración.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = resumen[resumen["lago"] == lago]
    eje.scatter(sub["nubosidad_pct"], sub["chl_media"], s=90,
                color=COLOR[lago], alpha=0.8)
    for _, f in sub.iterrows():
        eje.annotate(f"{f['fecha']:%b %y}", (f["nubosidad_pct"], f["chl_media"]),
                     textcoords="offset points", xytext=(6, 4), fontsize=7)
    if sub["nubosidad_pct"].nunique() > 2:
        r = sub[["nubosidad_pct", "chl_media"]].corr().iloc[0, 1]
        eje.set_title(f"{config.NOMBRE_LARGO[lago]}  (r = {r:+.2f})", loc="left")
    eje.set_xlabel("Nubosidad reportada de la escena (%)")
    eje.set_ylabel("Clorofila-a media (µg/L)")
    eje.grid(alpha=0.3)

fig.suptitle("¿La nubosidad explica el índice?", fontsize=13)
fig.tight_layout()
plt.show()